# Boolean Expression Evaluator using a LSTM model

In [1]:
import json
import itertools
from pathlib import Path
import ast

import numpy as np
import pandas as pd
import keras
import tensorflow as tf
from keras import ops
from sklearn.model_selection import train_test_split

print("Keras:", keras.__version__)
print("Backend:", keras.backend.backend())

Keras: 3.13.2
Backend: tensorflow


### Importing dataset into a pandas dataframe.

In [2]:
dataset = list(Path("/kaggle/input").rglob("1550k-boolean-exprs.jsonl"))[0]
with dataset.open('r') as f:
    df = pd.DataFrame(json.loads(line) for line in f if line)
print(f"Loaded {df.shape[0]} expressions.")
display(df.head())

Loaded 1550000 expressions.


,n,expr,table,depth
0,5,(~(~D & (A | B)) | ~((E | B) | ~C)),11111111001100110011101100110011,4
1,5,(((~D & (D | C)) & (A & (E | D))) | ((~D & (D ...,10101010111111111010111011111111,4
2,4,((((B & C) | ~C) | ~B) & (((B | A) | C) & (D &...,0001000100010001,4
3,5,((((C | B) | (D | B)) & (~E | (C | D))) & A),00000000000000000011111110111111,4
4,5,((((A & E) & ~A) | ~(D & C)) | (~~D & ~(C | B))),11111100111111001111110011111100,4


#### A lookup table is created for every unique token in expression and assigning an integer to it 
#### X is encoded by using the lookup table and padding using 0 to the size of the maximum length expression
#### y is encoded by casting every position to a float and padding using -1 to the size of the maximum length truth table.

In [5]:
lookup_table = {
    char: i + 1
    for i, char in enumerate(sorted(set("".join(df["expr"]))))
}


max_exprs_len = int(df["expr"].str.len().max())
max_truth_len = 2**df["n"].max()

def encode_X(exprs, max_expr_len, lookup_table):
    X = np.zeros((len(exprs), max_expr_len), dtype=np.int32)
    for i, expr in enumerate(exprs):
        X[i, :len(expr)] = [lookup_table[token] for token in expr]
    return X

def encode_y(truths, max_truth_len):
    y = np.full((len(truths), max_truth_len), -1.0, dtype=np.float32)
    for i, truth in enumerate(truths):
        y[i, :len(truth)] = [float(bit) for bit in truth]
    return y


X = encode_X(df["expr"].tolist(), max_exprs_len, lookup_table)
y = encode_y(df["table"].tolist(), max_truth_len)

print(f"Lookup: {lookup_table}")

Lookup: {' ': 1, '&': 2, '(': 3, ')': 4, 'A': 5, 'B': 6, 'C': 7, 'D': 8, 'E': 9, '|': 10, '~': 11}


#### A train and test set is created for effective evaluation of model on untrained samples.

In [6]:
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42)
print(f"Train size: {X_train.shape[0]}")
print(f"Test size: {X_test.shape[0]}")

Train size: 1240000
Test size: 310000


#### Custom loss and accuracy functions are created to evaluate binary accuracy without penalizing masked bits

In [7]:
def masked_binary_crossentropy(y_true, y_pred):
    mask = ops.cast(y_true >= 0.0, "float32")
    targets = ops.maximum(y_true, 0.0)
    eps = keras.backend.epsilon()
    probabilities = ops.clip(y_pred, eps, 1.0 - eps)
    elementwise_bce = -(
        targets * ops.log(probabilities)
        + (1.0 - targets) * ops.log(1.0 - probabilities)
    )
    valid_count = ops.maximum(ops.sum(mask, axis=-1), 1.0)
    return ops.sum(elementwise_bce * mask, axis=-1) / valid_count

def hits(y_true, y_pred):
    targets = ops.maximum(y_true, 0.0)
    predicted_bits = ops.cast(y_pred >= 0.5, "float32")
    return ops.cast(ops.equal(targets, predicted_bits), "float32")

def masked_bitwise_accuracy(y_true, y_pred):
    mask = ops.cast(y_true >= 0.0, "float32")
    valid_count = ops.maximum(ops.sum(mask, axis=-1), 1.0)
    return ops.sum(hits(y_true, y_pred) * mask, axis=-1) / valid_count

def exact_match(y_true, y_pred):
    mask = ops.cast(y_true >= 0.0, "float32")
    return ops.min(ops.maximum(hits(y_true, y_pred), 1.0 - mask), axis=-1)

#### A bidirectional LSTM model with a single hidden layer is created

In [8]:
model = keras.Sequential([
    keras.layers.Input(shape=(max_exprs_len,)),
    keras.layers.Embedding(input_dim=len(lookup_table) + 1, output_dim=16, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(250, return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(250)),
    keras.layers.Dense(400, activation="relu"),
    keras.layers.Dense(max_truth_len, activation="sigmoid")
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3, clipnorm=1.0),
    loss=masked_binary_crossentropy,
    metrics=[masked_bitwise_accuracy, exact_match],
)
model.summary()

I0000 00:00:1794031381.567971      47 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1794031381.571147      47 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 91, 16)         │           192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 91, 500)        │       534,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 500)            │     1,502,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 400)            │       200,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │        12,832 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,249,424 (8.58 MB)

 Trainable params: 2,249,424 (8.58 MB)

 Non-trainable params: 0 (0.00 B)

#### Model is fitted with appropriate callbacks to stop training early and effectively

In [9]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="exact_match", mode="max", patience=10, restore_best_weights=True,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="loss", factor=0.5, patience=3, min_lr=1e-5,
    ),
    keras.callbacks.ModelCheckpoint(
        "checkpoint_model.keras", monitor="exact_match", mode="max", save_best_only=True,
    ),
]
model.fit(
    X_train, y_train,
    batch_size=1024,
    epochs=10,
    callbacks=callbacks,
    shuffle=True
)

Epoch 1/10
1211/1211 ━━━━━━━━━━━━━━━━━━━━ 388s 315ms/step - exact_match: 0.1151 - loss: 0.4077 - masked_bitwise_accuracy: 0.7905 - learning_rate: 0.0010
Epoch 2/10
1211/1211 ━━━━━━━━━━━━━━━━━━━━ 392s 324ms/step - exact_match: 0.6544 - loss: 0.0675 - masked_bitwise_accuracy: 0.9731 - learning_rate: 0.0010
Epoch 3/10
1211/1211 ━━━━━━━━━━━━━━━━━━━━ 392s 324ms/step - exact_match: 0.9147 - loss: 0.0122 - masked_bitwise_accuracy: 0.9960 - learning_rate: 0.0010
Epoch 4/10
1211/1211 ━━━━━━━━━━━━━━━━━━━━ 392s 324ms/step - exact_match: 0.9674 - loss: 0.0057 - masked_bitwise_accuracy: 0.9985 - learning_rate: 0.0010
Epoch 5/10
1211/1211 ━━━━━━━━━━━━━━━━━━━━ 392s 324ms/step - exact_match: 0.9917 - loss: 0.0011 - masked_bitwise_accuracy: 0.9997 - learning_rate: 0.0010
Epoch 6/10
1211/1211 ━━━━━━━━━━━━━━━━━━━━ 392s 324ms/step - exact_match: 0.9898 - loss: 0.0014 - masked_bitwise_accuracy: 0.9996 - learning_rate: 0.0010
Epoch 7/10
1211/1211 ━━━━━━━━━━━━━━━━━━━━ 392s 324ms/step - exact_match: 0.9953 - 

#### Model is tested on untrained dataset to calculate its accuracy

In [10]:
test_pred = model.predict(X_test, verbose=0) >= 0.5
mask = y_test >= 0

actual = y_test == 1
correct = test_pred == actual

bitwise = correct[mask].mean()
exact = np.all(correct | ~mask, axis=1).mean()

print(f"Test bitwise accuracy: {bitwise:.2%}")
print(f"Test exact-table accuracy: {exact:.2%}")
print(f"Test expressions: {len(y_test):,}")

Test bitwise accuracy: 99.99%
Test exact-table accuracy: 99.84%
Test expressions: 310,000


#### 

In [14]:
def compare_truth_table(expr: str, model) -> pd.DataFrame:
    tree = ast.parse(expr.strip(), mode="eval").body
    variables = sorted({x.id for x in ast.walk(tree) if isinstance(x, ast.Name)})

    if not 1 <= len(variables) <= 5:
        raise ValueError("Expression must contain 1–5 variables.")

    def canonical(node, names):
        if isinstance(node, ast.Name):
            return names[node.id]
        if isinstance(node, ast.UnaryOp) and isinstance(node.op, (ast.Invert, ast.Not)):
            return "~" + canonical(node.operand, names)
        if isinstance(node, ast.BinOp) and isinstance(node.op, (ast.BitAnd, ast.BitOr)):
            op = "&" if isinstance(node.op, ast.BitAnd) else "|"
            a, b = sorted((canonical(node.left, names), canonical(node.right, names)))
            return f"({a} {op} {b})"
        raise ValueError("Use variables with &, |, ~, and parentheses.")

    names = dict(zip(variables, "ABCDE"))
    model_expr = canonical(tree, names)
    # model_expr = canonical(tree, {v: v for v in variables})

    def evaluate(node, env):
        if isinstance(node, ast.Name):
            return env[node.id]
        if isinstance(node, ast.UnaryOp):
            return not evaluate(node.operand, env)
        a, b = evaluate(node.left, env), evaluate(node.right, env)
        if isinstance(node.op, ast.BitAnd):
            return a and b
        if isinstance(node.op, ast.BitOr):
            return a or b
        raise ValueError("Unsupported operator.")

    rows = list(itertools.product([False, True], repeat=len(variables)))
    actual = [evaluate(tree, dict(zip(variables, bits))) for bits in rows]

    probs = model.predict(encode_X([model_expr], max_exprs_len, lookup_table), verbose=0)[0][:len(rows)]
    predicted = probs >= 0.5

    result = pd.DataFrame(rows, columns=variables)
    result["Actual"] = actual
    result["Predicted"] = predicted
    result["Probability"] = probs
    result["Correct"] = result["Actual"] == result["Predicted"]

    print(f"Model input: {model_expr}")
    print("Exact match" if result["Correct"].all()
          else f"{(~result['Correct']).sum()} of {len(rows)} rows wrong")

    return result

In [22]:
compare_truth_table("((((C & C) & ~B) | ((C & D) | C)) | (C | ((E | ~~(B | D)) & ~E)))", model)

Model input: ((((B & B) & ~A) | ((B & C) | B)) | (((D | ~~(A | C)) & ~D) | B))
Exact match


,B,C,D,E,Actual,Predicted,Probability,Correct
0,False,False,False,False,False,False,8.073356e-08,True
1,False,False,False,True,False,False,2.937900e-07,True
2,False,False,True,False,True,True,9.999915e-01,True
3,False,False,True,True,False,False,6.467592e-08,True
4,False,True,False,False,True,True,9.999996e-01,True
5,False,True,False,True,True,True,1.000000e+00,True
6,False,True,True,False,True,True,1.000000e+00,True
7,False,True,True,True,True,True,1.000000e+00,True
8,True,False,False,False,True,True,9.999983e-01,True
9,True,False,False,True,False,False,9.984231e-05,True


#### Model is saved to disk along with preprocessing encoding details 

In [25]:
model.save("LSTM_v2.keras")

with open("LSTM_preprocessing.json", "w", encoding="utf-8") as f:
    json.dump(
        {"lookup_table": lookup_table, "max_exprs_len": max_exprs_len, "max_truth_len": max_truth_len},
        f, ensure_ascii=False, indent=2,
    )

print("Saved LSTM_v2.keras and LSTM_preprocessing.json")

Saved LSTM_v2.keras and LSTM_preprocessing.json
